# 1 — Basic Local RAG Agent
## Embed, Store, Retrieve — ChromaDB in 30 Lines
⏱ ~10 min

**Retrieval-Augmented Generation (RAG)** grounds an LLM in real documents —
cutting hallucinations and enabling answers from private or fresh knowledge.
This notebook teaches the *simplest possible* production-grade RAG loop:
a LangGraph agent that retrieves from a local ChromaDB vector store and
generates answers in one tight cycle.

By the end you will understand every node, edge, and state transition in the
graph — and be able to swap in your own documents.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — What is RAG and why does it exist? |
| 2 | **LangGraph Primer** — Nodes, edges, state, and the ReAct loop |
| 3 | **The `@tool` decorator** — How the LLM learns about tools |
| 4 | **Building the retriever** — ChromaDB + inline docs (self-contained) |
| 5 | **Compiling the graph** — `StateGraph`, `MemorySaver`, `bind_tools` |
| 6 | **Running queries** — Single-turn and multi-turn with `thread_id` |
| 7 | **Debugging** — Inspecting message traces and tool call flow |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with the repo's `requirements.txt` installed (or run on Colab below)
- An `OPENAI_API_KEY` in `.env` or Colab Secrets

### Key References
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401  
> Yao, S., Zhao, J., et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models.* https://arxiv.org/abs/2210.03629  
> LangGraph documentation: https://langchain-ai.github.io/langgraph/  
> ChromaDB documentation: https://docs.trychroma.com

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "langchain-text-splitters",
            "chromadb",
            "python-dotenv",
            "langgraph",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), name = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid. "
    "Local: add it to .env. Colab: add it to Secrets."
)
print(f"OPENAI_API_KEY present: True ({key[:8]}...)")

---

## Part 1 — What Is RAG and Why Does It Exist?

---

### The Problem

Large language models are trained on a static snapshot of text (knowledge cutoff).
They have **no access** to:
- Your private documents, databases, or notes
- Information published after their training cutoff
- Real-time or domain-specific knowledge

When asked about these topics, the model **hallucinates** — producing
plausible-sounding but fabricated answers. RAG is the primary production solution.

---

### Three Approaches Compared

| Approach | Cost | Latency | Private docs | Fresh data |
|----------|------|---------|--------------|------------|
| **Prompt stuffing** (put all docs in context) | High (token cost) | High | Yes | Yes |
| **Fine-tuning** | Very high (training run) | Low | Partial | No |
| **RAG** | Low (retrieval + generation) | Medium | Yes | Yes |

RAG wins for most production use-cases: it is cheap, keeps knowledge fresh without
retraining, and is auditable — you can see exactly which chunks the model used.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Chunk** | A small excerpt of a document (typically 100–500 words) |
| **Embedding** | A dense numeric vector representing the *meaning* of text |
| **Vector store** | A database that stores embeddings and supports similarity search |
| **Retriever** | A component that finds the top-k most relevant chunks for a query |
| **ReAct loop** | Reason + Act: the agent decides whether to call a tool, then acts |
| **tool_calls** | The structured output an LLM emits when it wants to invoke a tool |
| **MemorySaver** | LangGraph's in-process checkpointer that persists state across turns |

### RAG + ReAct Architecture

This example combines two patterns: RAG (retrieve before generating) and ReAct
(the agent decides *when* to retrieve). The diagram shows both the offline indexing
phase and the online query phase inside a LangGraph state machine.

```
OFFLINE — run once when documents change
────────────────────────────────────────────────────────

  Your documents (inline, files, URLs, PDFs ...)
       |
       v
  +----------------------+
  | RecursiveCharacter   |  splits text into overlapping
  |   TextSplitter       |  chunks (e.g. 200 chars, 30 overlap)
  +----------+-----------+
             |
             v
  +----------------------+
  |  OpenAIEmbeddings    |  converts each chunk to a
  |  text-embedding-3-*  |  numeric vector  [0.12, -0.7, ...]
  +----------+-----------+
             |
             v
  +----------------------+
  |  ChromaDB (local)    |  stores vectors + text in memory
  +----------------------+  (or on disk with PersistentClient)


ONLINE — every user query (LangGraph ReAct loop)
────────────────────────────────────────────────────────

  User message
       |
       v
  +----------------+
  |   START node   |
  +-------+--------+
          |
          v
  +----------------+   no tool_calls
  |  agent node    | ─────────────────► END
  | (ChatOpenAI    |
  |  + bind_tools) |
  +-------+--------+
          | tool_calls present
          v
  +----------------+
  |  tools node    |  ToolNode executes retrieve_context():
  |  (ToolNode)    |  query -> ChromaDB -> top-k chunks -> string
  +-------+--------+
          | result added to messages
          |
          +──────────────► back to agent node
                           (now with context in messages)
```

> **Source**: ReAct loop pattern from Yao et al. (2022); RAG pipeline from Lewis et al. (2020).

---

## Part 2 — LangGraph Primer: Nodes, Edges, and State

---

LangGraph is a library for building stateful, cyclic agent workflows as directed
graphs. Each **node** is a Python function. **Edges** connect them. **State** is a
typed dict shared across all nodes in a single run.

### Core Concepts

| Concept | What it is |
|---------|------------|
| `StateGraph` | The graph builder; defines nodes and edges |
| `MessagesState` | Built-in state type with a `messages` list that auto-accumulates |
| `START` / `END` | Special sentinel nodes marking entry and exit |
| `add_node(name, fn)` | Register a Python function as a node |
| `add_edge(a, b)` | Always go from a to b |
| `add_conditional_edges(a, fn)` | Call `fn(state)` to decide which node comes next |
| `.compile(checkpointer=...)` | Freezes the graph into a runnable; wires up persistence |
| `MemorySaver` | In-process checkpointer — persists `messages` between `.invoke()` calls |
| `thread_id` | A key in `config` that namespaces memory slots |

### Why `MessagesState`?

`MessagesState` gives every node a list called `messages`. When a node returns
`{"messages": [new_msg]}`, LangGraph **appends** the new message to the existing
list — it does not replace it. This is the accumulator pattern that makes
multi-turn memory work without any extra bookkeeping.

### The ReAct Decision Loop

```
START
  |
  v
agent  ---- should_continue() ---->  "tools"  ---->  agent  (loop)
              |
              +---------------------->  END  (no more tool calls)
```

`should_continue` is a plain Python function that inspects `state["messages"][-1]`.
If the last message has `tool_calls`, route to `"tools"`. Otherwise, end.

---

## Part 3 — The `@tool` Decorator and the Retriever

---

The `@tool` decorator does two things:
1. The **docstring** becomes the tool description the LLM reads when deciding
   whether to call it. Write it as if you are explaining to the model what the
   function does — because you are.
2. The **function signature** (`query: str`) becomes the JSON schema the LLM
   uses to construct the call. Every parameter needs a type annotation.

### Live-URL vs Inline Docs

The original `src/utils.py` fetches live Python docs from the web on every tool
call (via `UnstructuredURLLoader`). This is realistic for a production agent but
slow and network-dependent in a workshop. This notebook uses **inline documents**
so everything is self-contained and instant.

### Tool Docstring Quality Matters

| Docstring quality | LLM behaviour |
|------------------|---------------|
| Too vague: `"Search for relevant documents."` | LLM may never call it, or call it for everything |
| Too narrow: `"Search for Python list slicing docs."` | LLM only calls it for slicing questions |
| Just right: `"Search the Python lists knowledge base..."` + specific examples | LLM calls it precisely when needed |

In [ ]:
# ─── 3-A: Build the retriever over inline documents ──────────────────────────
# We define SAMPLE_DOCS here rather than fetching URLs so the notebook is
# fully self-contained. In production, swap these for UnstructuredURLLoader,
# PyPDFLoader, or any LangChain document loader.

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

SAMPLE_DOCS = [
    Document(
        page_content=(
            "A Python list is an ordered, mutable collection of items. "
            "Create one with square brackets: my_list = [1, 2, 3]. "
            "Lists support indexing (my_list[0] returns 1) and negative "
            "indexing (my_list[-1] returns the last element)."
        )
    ),
    Document(
        page_content=(
            "List methods: append(item) adds to end, extend(iterable) adds "
            "multiple items, insert(i, item) adds at position i, remove(item) "
            "removes first match, pop() removes and returns last item, "
            "sort() sorts in place, reverse() reverses in place."
        )
    ),
    Document(
        page_content=(
            "List slicing: my_list[start:stop:step]. Examples: "
            "my_list[1:3] returns elements at index 1 and 2. "
            "my_list[::-1] reverses the list. "
            "my_list[::2] returns every other element starting from index 0."
        )
    ),
    Document(
        page_content=(
            "List comprehensions create new lists in one line: "
            "squares = [x**2 for x in range(10)]. "
            "Add a filter: evens = [x for x in range(20) if x % 2 == 0]. "
            "They are generally faster than equivalent for-loops."
        )
    ),
    Document(
        page_content=(
            "Common list functions: len(lst) returns length, "
            "sorted(lst) returns a new sorted list without modifying the original, "
            "min(lst) and max(lst) return extremes, "
            "sum(lst) totals numeric items, "
            "list(iterable) converts any iterable to a list."
        )
    ),
    Document(
        page_content=(
            "Nested lists (2D lists): matrix = [[1,2,3],[4,5,6],[7,8,9]]. "
            "Access element: matrix[row][col], e.g. matrix[1][2] returns 6. "
            "Flatten with a comprehension: flat = [x for row in matrix for x in row]."
        )
    ),
]

# Split and embed — this is the offline indexing step
_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
_chunks = _splitter.split_documents(SAMPLE_DOCS)
_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
_store = Chroma.from_documents(_chunks, _embeddings, collection_name="python_lists")
_retriever = _store.as_retriever(search_kwargs={"k": 2})

print(f"Indexed {len(_chunks)} chunks into ChromaDB (in-memory)")
print()
print("Sample retrieval test — query: 'how to reverse a list'")
test_results = _retriever.invoke("how to reverse a list")
for i, doc in enumerate(test_results, 1):
    print(f"  [{i}] {doc.page_content[:100]}...")

In [ ]:
# ─── 3-B: Define the tool with a descriptive docstring ───────────────────────
# The docstring IS the tool description sent to the LLM.
# A poor docstring means the LLM won't know when to call this tool.

@tool
def retrieve_context(query: str) -> str:
    """Search the Python lists knowledge base for relevant documentation.
    Use this tool whenever the user asks about Python lists, list methods,
    slicing, list comprehensions, nested lists, or related list operations.
    """
    docs = _retriever.invoke(query)
    if not docs:
        return "No relevant documentation found."
    return "\n\n".join(d.page_content for d in docs)


# Inspect what the tool looks like to the LLM
print("Tool name    :", retrieve_context.name)
print("Tool desc    :", retrieve_context.description)
print("Tool args    :", retrieve_context.args)
print()

# Manual invocation test (bypassing the LLM entirely)
result = retrieve_context.invoke({"query": "list comprehension examples"})
print("Manual tool test result (first 200 chars):")
print(result[:200], "...")

---

## Part 4 — Building the LangGraph

---

### `bind_tools` — Why It Matters

`model.bind_tools(tools)` sends the tool schemas to the model so it knows they
exist and how to call them. Without this, the model generates plain text — no
structured `tool_calls` output is produced.

### `ToolNode` — The Dispatch Layer

`ToolNode(tools)` is a pre-built LangGraph node that:
1. Reads `tool_calls` from the last AI message
2. Looks up the matching function by name
3. Calls it with the provided arguments
4. Wraps the result in a `ToolMessage` and appends it to `messages`

Without `ToolNode` you would write this dispatch logic yourself with `if/elif` chains.

### Graph Structure Summary

```python
workflow.add_edge(START, "agent")                        # always start with agent
workflow.add_conditional_edges("agent", should_continue) # decide: tools or END
workflow.add_edge("tools", "agent")                      # always return to agent
```

In [ ]:
# ─── 4-A: Assemble the graph ──────────────────────────────────────────────────
from typing import Literal

from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

# Wire the tool list — same list passed to both ToolNode and bind_tools
tools = [retrieve_context]
tool_node = ToolNode(tools)

# bind_tools injects the tool schemas into every LLM call
model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)


def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """Route: if the last message has tool_calls, go to tools. Otherwise finish."""
    last = state["messages"][-1]
    if last.tool_calls:
        return "tools"
    return END


def call_model(state: MessagesState) -> dict:
    """Invoke the LLM with the current message history."""
    response = model.invoke(state["messages"])
    # Return as a list — LangGraph appends to MessagesState, not replaces
    return {"messages": [response]}


# Build the graph
workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)  # ReAct decision point
workflow.add_edge("tools", "agent")                       # cycle: tool result -> agent

# Compile with MemorySaver — enables multi-turn conversation via thread_id
app = workflow.compile(checkpointer=MemorySaver())

print("Graph compiled successfully.")
print("Nodes:", list(workflow.nodes.keys()))

In [ ]:
# ─── 4-B: Visualise the graph ─────────────────────────────────────────────────
# Requires graphviz or the mermaid renderer. Falls back gracefully if unavailable.
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualisation unavailable:", e)
    print()
    print("Graph structure (text fallback):")
    print("  START -> agent")
    print("  agent -> tools  (when tool_calls present)")
    print("  agent -> END    (when no tool_calls)")
    print("  tools -> agent  (always, with tool result)")

---

## Part 5 — Running Queries

---

### How `.invoke()` Works

```python
result = app.invoke(
    {"messages": [HumanMessage(content="your question")]},
    config={"configurable": {"thread_id": 42}},
)
```

- `{"messages": [...]}` is the **initial state** for this turn. LangGraph appends
  to the checkpoint history associated with `thread_id`.
- `config["configurable"]["thread_id"]` namespaces memory. Same ID = same
  conversation. Different ID = fresh start.
- The return value is the **final state** — a dict with `messages` containing
  the full conversation history for this thread.

### Message Types

| Class | Role in `messages` |
|-------|--------------------|
| `HumanMessage` | User input |
| `AIMessage` | LLM response; may contain `tool_calls` |
| `ToolMessage` | Result returned by a tool execution |
| `SystemMessage` | System prompt (optional, prepend to messages list) |

In [ ]:
# ─── 5-A: First query — watch the full message trace ─────────────────────────
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": 42}}

result = app.invoke(
    {"messages": [HumanMessage(content="Explain what a Python list is")]},
    config=config,
)

print("=== Full message trace ===")
for msg in result["messages"]:
    role = type(msg).__name__
    content = str(msg.content)[:200] if msg.content else ""
    tool_info = ""
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        tool_info = f" [calls: {[tc['name'] for tc in msg.tool_calls]}]"
    print(f"[{role}]{tool_info} {content}")
    print()

In [ ]:
# ─── 5-B: Multi-turn conversation — same thread_id, follow-up question ────────
# MemorySaver persists the full conversation history per thread_id.
# The agent remembers the previous exchange without us re-sending any history.

result2 = app.invoke(
    {"messages": [HumanMessage(
        content="Show me a list comprehension example using what you just explained."
    )]},
    config=config,  # same thread_id = same memory slot
)

print("Follow-up answer:")
print(result2["messages"][-1].content)

In [ ]:
# ─── 5-C: Different thread — fresh conversation ───────────────────────────────
# A new thread_id starts from zero memory — no context from thread_id=42.

fresh_config = {"configurable": {"thread_id": 99}}

result3 = app.invoke(
    {"messages": [HumanMessage(
        content="What Python topic did we just discuss?"
    )]},
    config=fresh_config,
)

print("Response on a fresh thread:")
print(result3["messages"][-1].content)
print()
print("(Expected: the agent has no memory of the previous conversation.)")

---

## Part 6 — Notebook vs Production: Design Differences

---

The original `src/utils.py` fetches live web URLs on every tool call. That is
realistic for a demo but has design problems. Here is the comparison:

| Concern | Notebook (inline docs) | Original `src/utils.py` |
|---------|----------------------|-------------------------|
| **Data source** | Hard-coded `SAMPLE_DOCS` | `UnstructuredURLLoader` from 3 URLs |
| **When indexed** | Once at notebook start | On every single tool call |
| **Network required** | No | Yes |
| **Chunk size** | 200 chars, 30 overlap | 100 chars, 50 overlap |
| **Tool docstring** | Specific — names lists, slicing, etc. | Vague — `"Search for relevant documents."` |
| **Persistence** | In-memory (lost on restart) | In-memory (lost on restart) |

### The Right Production Pattern

Build the index **once** at startup, store it on disk with
`chromadb.PersistentClient(path="./chroma_db")`, and reuse the retriever for
every query. Never re-index inside the tool function.

In [ ]:
# ─── 6-A: Helper — build a reusable retriever (production pattern) ────────────

def build_retriever(docs, chunk_size=300, chunk_overlap=30, k=3):
    """Build and return a retriever from a list of Documents.
    Call this ONCE at startup, then pass the result into your @tool.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    store = Chroma.from_documents(
        chunks,
        OpenAIEmbeddings(model="text-embedding-3-small"),
    )
    return store.as_retriever(search_kwargs={"k": k})


# Build once at startup — the tool closes over this variable
RETRIEVER = build_retriever(SAMPLE_DOCS)


@tool
def retrieve_context_v2(query: str) -> str:
    """Search the Python lists knowledge base for relevant documentation.
    Use this whenever the user asks about Python lists, slicing, comprehensions,
    list methods (append, pop, sort, etc.), or nested lists.
    """
    docs = RETRIEVER.invoke(query)
    return "\n\n".join(d.page_content for d in docs) or "No results found."


print("Improved tool test — 'how to sort a list':")
print(retrieve_context_v2.invoke({"query": "how to sort a list"})[:250])

---

## Part 7 — Debugging and Inspecting the Agent

---

### Debugging Checklist

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Agent never calls the tool | Docstring too vague | Improve the docstring |
| Agent calls tool but answer is wrong | Wrong chunks retrieved | Adjust `k` or chunk size |
| Agent loops forever | `should_continue` always returns `"tools"` | Check routing logic |
| Multi-turn forgets context | Wrong `thread_id` | Use the same config dict |
| `AssertionError: OPENAI_API_KEY` | Key not loaded | Check `.env` or Colab Secrets |

In [ ]:
# ─── 7-A: Pretty-print the full message trace ─────────────────────────────────

def pretty_trace(messages):
    """Print a human-readable trace of a LangGraph message list."""
    divider = "─" * 60
    for msg in messages:
        role = type(msg).__name__
        print(divider)
        print(f"[{role}]")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print(f"  tool_calls : {[tc['name'] for tc in msg.tool_calls]}")
            for tc in msg.tool_calls:
                print(f"  args       : {tc.get('args', {})}")
        if msg.content:
            content = str(msg.content)
            print(content[:400])
            if len(content) > 400:
                print(f"  ... [{len(content) - 400} more chars]")
    print(divider)


debug_result = app.invoke(
    {"messages": [HumanMessage(content="What list methods can I use to add items?")]},
    config={"configurable": {"thread_id": 200}},
)
print("Full trace for: 'What list methods can I use to add items?'\n")
pretty_trace(debug_result["messages"])

In [ ]:
# ─── 7-B: Observe the tool-skip decision ─────────────────────────────────────
# A question the LLM can answer from training data (not Python-list-specific)
# should NOT trigger a tool call — the agent answers directly.

no_tool_result = app.invoke(
    {"messages": [HumanMessage(content="What is 7 multiplied by 8?")]},
    config={"configurable": {"thread_id": 300}},
)

messages = no_tool_result["messages"]
tool_was_called = (
    len(messages) > 2
    or (hasattr(messages[1], "tool_calls") and bool(messages[1].tool_calls))
)
print(f"Message count : {len(messages)}")
print(f"Tool called   : {tool_was_called}")
print(f"Answer        : {messages[-1].content}")
print()
print("Expected: 2 messages (HumanMessage + AIMessage) — no ToolMessage.")

---

## Exercises

---

### Exercise 1 — Change the Knowledge Domain

Replace `SAMPLE_DOCS` with 5–6 documents about Python **dictionaries**
(creation, methods like `.get()`, `.items()`, `.update()`, dict comprehensions).
Update the `@tool` docstring to match, rebuild the retriever, and ask:
> "How do I iterate over both keys and values in a dictionary?"

---

### Exercise 2 — Observe the Tool-Skip Decision

Ask a question the LLM can answer without retrieval:
> "What is the capital of France?"

Does the graph call `retrieve_context`? Inspect the message trace.
What does this tell you about how `bind_tools` works?

---

### Exercise 3 — Add a Second Tool

Create a `@tool` function `word_count(text: str) -> str` that returns the number
of words in a string. Add it to the `tools` list and recompile the graph.
Ask: "How many words are in the phrase 'Python lists are ordered mutable collections'?"

---

### Exercise 4 — Thread Isolation

Run the follow-up question "Show me a list comprehension from what you explained"
on a **new** `thread_id` with no prior context. What happens? Does the agent
retrieve context or answer from training knowledge?

In [ ]:
# ── Exercise 1 Starter — Change the Knowledge Domain ─────────────────────────

MY_DOCS = [
    Document(page_content="TODO: add content about Python dictionaries."),
    # Add 4 more documents here...
]

# TODO: update the docstring to describe the new domain
@tool
def retrieve_my_context(query: str) -> str:
    """TODO: describe what this tool searches — be specific!"""
    _my_retriever = build_retriever(MY_DOCS)
    docs = _my_retriever.invoke(query)
    return "\n\n".join(d.page_content for d in docs) or "No results."


# TODO: rebuild the graph with the new tool and run a query
# my_tools = [retrieve_my_context]
# my_model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(my_tools)
# ...

In [ ]:
# ── Exercise 3 Starter — Add a Second Tool ────────────────────────────────────

@tool
def word_count(text: str) -> str:
    """TODO: add a description. Count the number of words in the given text."""
    # TODO: implement — hint: len(text.split())
    pass


# TODO: build a new graph with [retrieve_context, word_count] and run a query
# two_tools = [retrieve_context, word_count]
# ...

---

## What's Next?

You now understand the foundational LangGraph RAG loop. Here is where to go next:

### Deepen the RAG knowledge
- **Example 12 — Basic RAG Notebook** (`../12-basic-rag-notebook/rag_workbook.ipynb`):
  Full deep-dive into text splitting, embeddings, ChromaDB persistence, LCEL chains,
  real document loaders (PDFs, web pages), and debugging — the complete RAG curriculum.

### Extend this agent
- **Example 4 — Basic ReAct Agent** (`../4-basic-react-agent/`): same ReAct loop
  with multiple tools and a system prompt.
- **Example 10 — Streaming RAG** (`../10-streaming-rag/`): stream graph updates
  node-by-node instead of blocking until `END`.
- **Example 13 — Structured Output** (`../13-structured-output/`): LLM extracts
  typed Pydantic fields from retrieved documents.

### Evaluate what you built
- **Example 16 — RAG Evaluation with RAGAS** (`../16-rag-eval-notebook/rag_eval_workbook.ipynb`):
  score your pipeline on Faithfulness, Answer Relevance, and Context Recall.

### Further reading
- Lewis et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
- Yao et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models.* https://arxiv.org/abs/2210.03629
- Gao et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey.* https://arxiv.org/abs/2312.10997
- LangGraph concepts: https://langchain-ai.github.io/langgraph/concepts/
- ChromaDB docs: https://docs.trychroma.com

---

*Workshop complete. The next step is Example 12 for a deep dive into every RAG component.*

---

## Exercise Answer Key

Use this section as a reference **after** attempting the exercises yourself.
These are sample solutions, not the only correct answers.

---

### Exercise 1 — Change the Knowledge Domain

The key changes are:
1. Replace `SAMPLE_DOCS` with dictionary-focused content
2. Update the `@tool` docstring to mention dictionaries, `.get()`, `.items()`, etc.
3. Rebuild the retriever from the new docs — the graph structure stays the same

A strong docstring explicitly lists the kinds of queries the tool handles. This
prevents the LLM from calling the tool when the user asks unrelated questions.

In [ ]:
# Answer Key — Exercise 1

DICT_DOCS = [
    Document(
        page_content=(
            "A Python dictionary is an unordered collection of key-value pairs. "
            "Create with curly braces: d = {'name': 'Alice', 'age': 30}. "
            "Keys must be hashable (strings, numbers, tuples)."
        )
    ),
    Document(
        page_content=(
            "Dictionary access: d['key'] raises KeyError if missing. "
            "Use d.get('key', default) to safely retrieve with a fallback. "
            "Use 'key' in d to check existence without raising."
        )
    ),
    Document(
        page_content=(
            "Dictionary methods: .keys() returns all keys, .values() returns all values, "
            ".items() returns (key, value) pairs. Use in a for loop: "
            "for k, v in d.items(): print(k, v)."
        )
    ),
    Document(
        page_content=(
            "Dictionary comprehensions: {k: v for k, v in pairs if condition}. "
            "Example: squares = {x: x**2 for x in range(5)}."
        )
    ),
    Document(
        page_content=(
            "Merging dictionaries: d1.update(d2) modifies d1 in place. "
            "Python 3.9+: merged = d1 | d2 creates a new merged dict. "
            "Python 3.5+: merged = {**d1, **d2} also works."
        )
    ),
]

DICT_RETRIEVER = build_retriever(DICT_DOCS)


@tool
def retrieve_dict_context(query: str) -> str:
    """Search the Python dictionaries knowledge base for relevant documentation.
    Use this tool when the user asks about Python dicts, keys, values,
    .get(), .items(), .keys(), .values(), dict comprehensions, or merging dicts.
    """
    docs = DICT_RETRIEVER.invoke(query)
    return "\n\n".join(d.page_content for d in docs) or "No results."


dict_tools = [retrieve_dict_context]
dict_model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(dict_tools)


def call_dict_model(state: MessagesState) -> dict:
    return {"messages": [dict_model.invoke(state["messages"])]}


dict_workflow = StateGraph(MessagesState)
dict_workflow.add_node("agent", call_dict_model)
dict_workflow.add_node("tools", ToolNode(dict_tools))
dict_workflow.add_edge(START, "agent")
dict_workflow.add_conditional_edges("agent", should_continue)
dict_workflow.add_edge("tools", "agent")
dict_app = dict_workflow.compile(checkpointer=MemorySaver())

dict_result = dict_app.invoke(
    {"messages": [HumanMessage(
        content="How do I iterate over both keys and values in a dictionary?"
    )]},
    config={"configurable": {"thread_id": 1}},
)
print(dict_result["messages"][-1].content)

### Exercise 2 — Observe the Tool-Skip Decision

When you ask "What is the capital of France?", the agent answers directly without
calling `retrieve_context`. Why?

`bind_tools` gives the model the **option** to call tools — it does not force it.
The model decides based on whether the question matches the tool's docstring.
`retrieve_context` says "Search the Python lists knowledge base" — geography
questions do not match, so the model answers from training knowledge.

**Key insight:** A well-written tool docstring is the primary signal the LLM uses
for routing. Vague docstrings lead to either missing calls (the tool is never used)
or spurious calls (the tool fires on unrelated questions).

In [ ]:
# Answer Key — Exercise 3 — Add a Second Tool

@tool
def word_count(text: str) -> str:
    """Count the number of words in a given string.
    Use this tool when the user asks to count words in a phrase or sentence.
    """
    count = len(text.split())
    return f"The text contains {count} word{'s' if count != 1 else ''}."


two_tools = [retrieve_context, word_count]
two_tool_node = ToolNode(two_tools)
two_model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(two_tools)


def call_two_model(state: MessagesState) -> dict:
    return {"messages": [two_model.invoke(state["messages"])]}


two_workflow = StateGraph(MessagesState)
two_workflow.add_node("agent", call_two_model)
two_workflow.add_node("tools", two_tool_node)
two_workflow.add_edge(START, "agent")
two_workflow.add_conditional_edges("agent", should_continue)
two_workflow.add_edge("tools", "agent")
two_app = two_workflow.compile(checkpointer=MemorySaver())

two_result = two_app.invoke(
    {"messages": [HumanMessage(
        content="How many words are in the phrase 'Python lists are ordered mutable collections'?"
    )]},
    config={"configurable": {"thread_id": 10}},
)
print(two_result["messages"][-1].content)

### Exercise 4 — Thread Isolation

Running the follow-up question on a new `thread_id` means `MemorySaver` has no
prior messages in that slot. The agent starts completely fresh.

Depending on the phrasing, the agent will either:
- Call `retrieve_context` because the question mentions Python lists, OR
- Answer from training knowledge if the phrasing is generic enough

**Key insight:** `thread_id` scopes memory to a conversation. Different IDs are
completely isolated — there is no shared memory between threads in `MemorySaver`.
For persistent cross-session memory, use `SqliteSaver` or `PostgresSaver` from
`langgraph.checkpoint.sqlite` and `langgraph.checkpoint.postgres` respectively.